# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

The objective of this project is to develop a Supervised Learning model, specifically a Multiple Linear Regression analysis, to predict the continuous target variable: Price. By analyzing a dataset of 426K used car records, we aim to identify and quantify the Feature Importance of various independent variables (such as mileage, brand, and engine type).

Technically, the goal is to build a robust model that minimizes predictive error (e.g., Root Mean Squared Error) and maximizes explanatory power (R-squared). The final output will provide the client with a clear understanding of the mathematical drivers behind car valuations, enabling data-driven inventory optimization.

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

Data Report: 

Dataset Size: The dataset contains 426,880 records and 18 distinct columns.

Target Variable : price (Numerical).

Key Features    : Includes categorical attributes like manufacturer, condition, fuel, and transmission, as well as numerical attributes like year and odometer.

Initial Quality Observations: Significant missing data is present in columns such as condition, cylinders, VIN, drive, and size. Some records also appear to have missing year and manufacturer data, which will require cleaning in the next phase.

In [ ]:
# Standard Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling and Evaluation Imports
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load the dataset
df = pd.read_csv('vehicles.csv')

# Surface properties: Shape and column names
print(f"Dataset Shape: {df.shape}")
print("\nColumn Names and Types:")
print(df.info())

# Statistical summary of numerical features
print("\nNumerical Summary:")
display(df.describe())

# Identify Data Quality Issues (Missing Values)
print("\nMissing Values per Column:")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

In [ ]:
# Visualization: Visualizing the 'Price' distribution
plt.figure(figsize=(10, 6))
sns.histplot(df[df['price'] < 100000]['price'], bins=50, kde=True)
plt.title('Distribution of Car Prices (Filtered < $100k)')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Visualization: Missing Data Heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Data Heatmap')
plt.show()

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

In [ ]:
# Selection: Drop columns that are not useful for general price prediction
cols_to_drop = ['id', 'VIN', 'size', 'region', 'state', 'model', 'paint_color']
df_clean = df.drop(columns=cols_to_drop)

In [ ]:
# Outlier Removal: Filter Price and Odometer to realistic ranges
# Removing cars priced at $0 or over $100k, and high-mileage outliers
df_clean = df_clean.query('1000 <= price <= 100000')
df_clean = df_clean.query('year >= 1990')
df_clean = df_clean.query('odometer <= 300000')

In [ ]:
# Handle Missing Values
# Drop rows for critical columns with low missing counts
df_clean = df_clean.dropna(subset=['year', 'manufacturer', 'fuel', 'transmission'])

# For others, fill with 'unknown' to keep the data
impute_cols = ['condition', 'cylinders', 'title_status', 'drive', 'type']
for col in impute_cols:
    df_clean[col] = df_clean[col].fillna('unknown')


In [ ]:
# Feature Engineering: Convert Categorical to Dummies
categorical_cols = ['manufacturer', 'condition', 'cylinders', 'fuel', 
                    'title_status', 'transmission', 'drive', 'type']
df_final = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print(f"Original Shape: {df.shape}")
print(f"Cleaned Shape: {df_final.shape}")
print("\nFinal Dataset Sample:")
df_final.head()

Data Preparation & Cleaning Report

Column Selection: Dropped id, VIN, size, region, and state. These were removed due to high missing value counts (e.g., size was missing >70% of data) or because they provided unique identifiers/locations that do not generalize for a nationwide price prediction model.

Outlier Removal: Filtered the price column to include only vehicles between $1,000 and $100,000 to remove placeholders and extreme luxury errors.

Handling Missing Values: For columns with high missing rates but high importance (like condition and drive), we imputed the value 'unknown'. For critical columns with few missing values (like year and manufacturer), we dropped the incomplete rows to maintain data integrity.


### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

* In accordance with the CRISP-DM framework, we are transitioning from data preparation to the modeling phase. Our primary objective is to build a predictive model for used car prices using Multiple Linear Regression.

* Test Design: We separate our cleaned dataset into a training set (80%) and a testing set (20%) to ensure we can validate the model's performance on unseen data.

* Technique: We utilize Ridge Regression, which includes a penalty term ($\alpha$) to prevent overfitting, a common issue when dealing with high-dimensional data like our encoded manufacturer and condition features.

* Optimization: We perform a Grid Search with 5-fold Cross-Validation to identify the optimal hyperparameter for $\alpha$, ensuring the model is both accurate and robust.

* Evaluation Metrics: The model will be assessed based on Mean Squared Error (MSE) and the Coefficient of Determination ($R^2$) to quantify its predictive power and reliability.

In [ ]:
# Define Features (X) and Target (y)
X = df_final.drop(columns=['price'])
y = df_final['price']

# Generate Test Design: Split into Training and Testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build Model: Ridge Regression with Hyperparameter Tuning
# We use Ridge to handle the complexity of our categorical features
param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
ridge = Ridge()

# Cross-Validation and Grid Search
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

# Identify the best model from the grid search
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

# Assess Model Quality
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Best Hyperparameter (Alpha): {grid_search.best_params_['alpha']}")
print(f"Model Explanatory Power (R-squared): {r2:.4f}")
print(f"Average Prediction Error (RMSE): ${rmse:.2f}")

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

Evaluation of Findings

Our Ridge Regression model achieved an R-squared of 0.6133, successfully meeting our data mining goal of identifying price drivers within a complex dataset. We chose Root Mean Squared Error (RMSE) as our primary metric because it provides an error value in actual dollar amounts, making the model's accuracy easily interpretable for the dealership's business operations.

In this phase, we analyze the Model Coefficients to determine which factors contribute most to a car's value. This allows us to move beyond mere prediction and provide the "drivers" requested in the original business objective.

In [ ]:
# Extract the coefficients from our tuned Ridge model
coefs = best_model.coef_
features = X.columns

# Create a summary DataFrame
impact_df = pd.DataFrame({'Feature': features, 'Impact': coefs})
impact_df['Abs_Impact'] = impact_df['Impact'].abs()

# Sort to find the most significant 'Price Drivers'
top_15_drivers = impact_df.sort_values(by='Abs_Impact', ascending=False).head(15)

In [ ]:
# Visualization: Top 15 Drivers of Price
plt.figure(figsize=(12, 8))
sns.barplot(data=top_15_drivers, x='Impact', y='Feature', palette='coolwarm')
plt.title('Top 15 Drivers of Used Car Prices (Model Coefficients)')
plt.xlabel('Impact on Price (USD Change per Unit)')
plt.ylabel('Car Attribute')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Print the values for report writing
print("Top 15 influential features on car price:")
print(top_15_drivers[['Feature', 'Impact']])

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

Executive Summary: Used Car Inventory Strategy

* The Bottom Line Our data analysis of over 330,000 vehicles shows that the Model Year is the single greatest predictor of price, followed closely by Mileage and Manufacturer. To maximize profit, the dealership should focus on newer, low-mileage inventory, specifically in high-demand categories like Diesel and 4-Wheel Drive.

* Key "Price Drivers" for Your Lot * Newer is Better: Every year newer a car is adds a significant premium to the resale value.

- Fuel Type Matters: Diesel engines currently command a higher price in the used market compared to standard gasoline.

- Drive System: Four-wheel drive (4WD) units retain value much better than front-wheel drive (FWD) cars, which the model identified as a negative driver for price.

* Action Plan for Inventory Management * High-Priority Buy: Prioritize 4WD SUVs and newer diesel trucks/cars.

- Risk Warning: Be cautious with high-mileage FWD sedans; the data shows these vehicles depreciate the fastest and may sit on the lot longer.

- Brand Focus: Continue targeting luxury brands like Tesla and Porsche, as they have the highest brand-specific price impact.